# reparameterization-trick — ex1: differentiable Gaussian sampling with gradient check

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `reparameterization-trick`. Running the final beacon cell reports progress against the `VAE: Reparameterization trick` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `VAE: Reparameterization trick` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`reparameterization-trick`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "reparameterization-trick"
DD_SUBTOPIC = "VAE: Reparameterization trick"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Reparameterization trick — quick refresher

The trick that makes VAEs trainable. Instead of sampling `z ~ N(mu, sigma^2)` directly (which is non-differentiable w.r.t. `mu` and `sigma`), we sample `eps ~ N(0, 1)` and compute:

```python
sigma = (0.5 * logsigma).exp()   # sigma = exp(logsigma / 2)
z = mu + sigma * eps             # eps = randn_like(mu)
```

Now gradients flow through `mu` (direct) and `sigma` (via the multiplicative path), and the randomness lives in `eps` — a constant from autograd's point of view.

**Why `0.5 * logsigma`.** The encoder emits log-VARIANCE, not log-standard-deviation. `sigma = sqrt(var) = sqrt(exp(logvar)) = exp(logvar / 2)`. The factor of 0.5 in the exponent IS the square root.

**Why `randn_like(mu)`.** Matches shape, dtype, and device automatically. `t.randn(*mu.shape)` would default to `float32` on CPU — breaks silently when `mu` is on GPU or `bfloat16`.

### Exercise 1 — differentiable Gaussian sampling with gradient check

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the reparameterization trick — `z = mu + (0.5 * logsigma).exp() * eps` with `eps = randn_like(mu)` — to draw a differentiable Gaussian sample from `N(mu, exp(logsigma))`.
> Keywords: reparam, randn_like, differentiable-sample, vae
> ```

**KCs targeted:** `reparam-formula`, `randn-like-noise`

Implement `ex1_reparameterize(mu, logsigma)`. The differentiable-sample trick that makes VAEs trainable end-to-end:

1. `mu` and `logsigma` both have shape `(B, latent_dim)` — outputs of the VAE encoder head.
2. Compute `sigma = (0.5 * logsigma).exp()` — i.e. `exp(logsigma / 2)`. (Treat `logsigma` as log-variance — the ARENA convention.)
3. Draw noise: `eps = t.randn_like(mu)`. Use `randn_like` (not `t.randn(*mu.shape)`) so dtype + device match `mu` automatically.
4. Return `z = mu + sigma * eps` — same shape as `mu`.

Input: `mu`, `logsigma` — `(B, latent_dim)` float tensors.
Output: `(B, latent_dim)` float tensor.

The visualization compares the empirical distribution of 2-D samples to the theoretical mean ± 2σ ellipse for several `(mu, logsigma)` choices.

In [ ]:
def ex1_reparameterize(mu: Tensor, logsigma: Tensor) -> Tensor:
    """z = mu + exp(logsigma / 2) * eps, eps ~ N(0, 1)."""
    raise NotImplementedError()


def _test_ex1():
    # Shape / dtype smoke test.
    B, latent_dim = 8, 4
    mu = t.zeros(B, latent_dim)
    logsigma = t.zeros(B, latent_dim)
    t.manual_seed(0)
    z = ex1_reparameterize(mu, logsigma)
    assert z.shape == (B, latent_dim), f'expected (B,latent_dim), got {tuple(z.shape)}'
    assert z.dtype == t.float32

    # Distribution sanity — with mu=0, logsigma=0 → sigma=1 → z ~ N(0, 1).
    t.manual_seed(0)
    big_mu = t.zeros(20000, 3)
    big_ls = t.zeros(20000, 3)
    z_big = ex1_reparameterize(big_mu, big_ls)
    assert abs(z_big.mean().item()) < 0.05, f'mean should be ~0, got {z_big.mean().item():.4f}'
    assert abs(z_big.std().item() - 1.0) < 0.05, f'std should be ~1, got {z_big.std().item():.4f}'

    # Distribution sanity — with mu=5, logsigma=log(4) → sigma=2 → z ~ N(5, 4).
    import math
    t.manual_seed(0)
    mu_shift = t.full((20000, 3), 5.0)
    ls_shift = t.full((20000, 3), math.log(4.0))    # variance=4 → sigma=2
    z_shift = ex1_reparameterize(mu_shift, ls_shift)
    assert abs(z_shift.mean().item() - 5.0) < 0.1, f'mean should be ~5, got {z_shift.mean().item():.4f}'
    assert abs(z_shift.std().item() - 2.0) < 0.1, f'std should be ~2, got {z_shift.std().item():.4f}'

    # Differentiability — gradient must flow back to mu and logsigma.
    g_mu = t.zeros(2, 3, requires_grad=True)
    g_ls = t.zeros(2, 3, requires_grad=True)
    t.manual_seed(7)
    g_z = ex1_reparameterize(g_mu, g_ls)
    g_z.sum().backward()
    assert g_mu.grad is not None and (g_mu.grad != 0).any(), 'mu must receive nonzero gradient'
    assert g_ls.grad is not None, 'logsigma must receive gradient'

    # --- Visualization: empirical samples vs theoretical mean ± 2σ ellipse ---
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    cases = [
        (t.tensor([0.0, 0.0]), t.tensor([0.0, 0.0]),       'mu=(0,0), sigma=(1,1)'),
        (t.tensor([2.0, -1.0]), t.tensor([math.log(0.25), math.log(1.0)]), 'mu=(2,-1), sigma=(0.5,1)'),
        (t.tensor([-1.0, 1.5]), t.tensor([math.log(4.0), math.log(0.25)]), 'mu=(-1,1.5), sigma=(2,0.5)'),
    ]
    for ax, (m, ls, title) in zip(axes, cases):
        t.manual_seed(0)
        mb = m.unsqueeze(0).expand(2000, -1)
        lb = ls.unsqueeze(0).expand(2000, -1)
        samples = ex1_reparameterize(mb, lb)
        ax.scatter(samples[:, 0].numpy(), samples[:, 1].numpy(), s=4, alpha=0.4, color='steelblue')
        # 2σ ellipse from theoretical params.
        sig = (0.5 * ls).exp()
        theta = t.linspace(0, 2 * math.pi, 100)
        ex = m[0].item() + 2 * sig[0].item() * theta.cos().numpy()
        ey = m[1].item() + 2 * sig[1].item() * theta.sin().numpy()
        ax.plot(ex, ey, 'r-', lw=2, label='theoretical 2σ')
        ax.scatter([m[0].item()], [m[1].item()], color='red', marker='x', s=80, label='theoretical mean')
        ax.set_title(title)
        ax.set_aspect('equal')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_reparameterize(mu: Tensor, logsigma: Tensor) -> Tensor:
    sigma = (0.5 * logsigma).exp()
    eps = t.randn_like(mu)
    return mu + sigma * eps
```

**Why the trick works.** Sampling `z ~ N(mu, sigma^2)` directly has no gradient — you can't differentiate through a random draw. By moving the randomness OUT of the path (into `eps`) and combining it deterministically with `mu` and `sigma`, the gradient flows through `mu` (the additive path) and `sigma` (the multiplicative path) — exactly the parameters we need to train.

**Why `(0.5 * logsigma).exp()` and not `logsigma.exp() ** 0.5`.** Mathematically identical (`exp(x/2) = sqrt(exp(x))`), but the former is one fewer op and numerically friendlier — no square root of a possibly-tiny number.

**`randn_like(mu)` is the right choice.** It matches dtype + device + shape in one call. `t.randn(*mu.shape, device=mu.device, dtype=mu.dtype)` is correct but verbose; bare `t.randn(*mu.shape)` silently breaks on GPU.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()